# Quick Start: CausalPy Overview

This notebook demonstrates all major CausalPy model types in one place.
Use it as a reference for choosing the right method.

**CausalPy model types covered:**
1. `InterruptedTimeSeries` — single-unit time series with a known intervention
2. `DifferenceInDifferences` — panel data, difference-in-differences estimator
3. `SyntheticControl` — panel data, weighted donor counterfactual
4. `RegressionDiscontinuity` — assignment determined by a running variable cutoff

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import causalpy as cp
import arviz as az

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

print(f"CausalPy version: {cp.__version__}")
print("Available model classes:")
for name in dir(cp):
    if not name.startswith("_"):
        print(f"  cp.{name}")

## Method Decision Guide

```
Do you have a single time series?
  YES → Is there a sharp intervention date?
          YES → InterruptedTimeSeries
          NO  → Other methods (ARIMA, etc.)
  NO  → Do you have panel data (multiple units)?
          YES → Is assignment determined by a cutoff score?
                  YES → RegressionDiscontinuity
                  NO  → Do you have many treated units?
                          YES → DifferenceInDifferences
                          NO  → SyntheticControl (one treated unit)
```

---

## 1. InterruptedTimeSeries

**Use when:** Single time series, known intervention date, 10+ pre-intervention observations.

**Causal assumption:** The pre-intervention trend would have continued without the intervention.

In [ ]:
# --- ITS: Smoking ban → AMI rates ---
n = 48
n_pre = 24
months = np.arange(n).astype(float)
treated = (months >= n_pre).astype(float)
t_post = np.maximum(months - n_pre, 0).astype(float)

ami_rate = (
    85.0 - 0.15 * months - 12.0 * treated - 0.10 * t_post * treated
    + np.random.normal(0, 4, n)
)

df_its = pd.DataFrame({"month": months, "ami_rate": ami_rate,
                        "treated": treated, "t_post": t_post})

its = cp.InterruptedTimeSeries(
    data=df_its,
    treatment_time=n_pre,
    formula="ami_rate ~ 1 + month + treated + t_post",
    model=cp.pymc_models.LinearRegression(
        sample_kwargs={"draws": 500, "tune": 500, "chains": 2,
                       "progressbar": False, "random_seed": 42}
    ),
)

# Key results
level_samples = its.idata.posterior["treated"].values.flatten()
slope_samples = its.idata.posterior["t_post"].values.flatten()
print("ITS Results")
print(f"  Level change: {level_samples.mean():.1f} "
      f"[{az.hdi(level_samples, 0.94)[0]:.1f}, {az.hdi(level_samples, 0.94)[1]:.1f}]")
print(f"  Slope change: {slope_samples.mean():.2f} "
      f"[{az.hdi(slope_samples, 0.94)[0]:.2f}, {az.hdi(slope_samples, 0.94)[1]:.2f}]")
print(f"  True values:  level=-12.0, slope=-0.10")

## 2. DifferenceInDifferences

**Use when:** Panel data, multiple treated and control units, pre-post design.

**Causal assumption:** Parallel trends — treated and control would have changed identically without treatment.

In [ ]:
# --- DiD: Policy intervention in half the cities ---
n_cities = 20
n_periods = 2  # Pre and post
true_did_effect = -5.0

cities = [f"City_{i:02d}" for i in range(n_cities)]
treated_cities = cities[:10]  # Half treated, half control

rows_did = []
for city in cities:
    is_treated = city in treated_cities
    baseline = np.random.normal(50, 5)
    common_time_trend = np.random.normal(2, 1)  # Shared pre-post trend
    for period in [0, 1]:  # 0=pre, 1=post
        effect = true_did_effect if (is_treated and period == 1) else 0.0
        outcome = baseline + common_time_trend * period + effect + np.random.normal(0, 2)
        rows_did.append({"city": city, "period": period, "treated": int(is_treated), "outcome": outcome})

df_did = pd.DataFrame(rows_did)

did = cp.DifferenceInDifferences(
    data=df_did,
    formula="outcome ~ 1 + treated + period + treated:period",
    time_variable_name="period",
    group_variable_name="treated",
    model=cp.pymc_models.LinearRegression(
        sample_kwargs={"draws": 500, "tune": 500, "chains": 2,
                       "progressbar": False, "random_seed": 42}
    ),
)

# DiD estimate: coefficient on treated:period interaction
did_samples = did.idata.posterior["treated:period"].values.flatten()
print("DiD Results")
print(f"  ATT estimate: {did_samples.mean():.1f} "
      f"[{az.hdi(did_samples, 0.94)[0]:.1f}, {az.hdi(did_samples, 0.94)[1]:.1f}]")
print(f"  True ATT: {true_did_effect}")

## 3. RegressionDiscontinuity

**Use when:** Assignment is based on whether a running variable exceeds a cutoff.

**Causal assumption:** Units just above and below the cutoff are comparable — local randomization.

In [ ]:
# --- RD: Scholarship cutoff at score = 0.5 ---
n_students = 200
true_rd_effect = 8.0  # Effect of receiving the scholarship

score = np.random.uniform(0, 1, n_students)  # Running variable
treated_rd = (score >= 0.5).astype(float)    # Treatment: score >= 0.5

# Outcome: GPA, affected by score (trend), treatment, and noise
gpa = (
    60.0 + 20.0 * score           # Continuous relationship with score
    + true_rd_effect * treated_rd  # Scholarship effect
    + np.random.normal(0, 3, n_students)
)

df_rd = pd.DataFrame({"score": score, "gpa": gpa, "treated": treated_rd})

rd = cp.RegressionDiscontinuity(
    data=df_rd,
    formula="gpa ~ 1 + score + treated",
    running_variable_name="score",
    treatment_threshold=0.5,
    model=cp.pymc_models.LinearRegression(
        sample_kwargs={"draws": 500, "tune": 500, "chains": 2,
                       "progressbar": False, "random_seed": 42}
    ),
)

rd_samples = rd.idata.posterior["treated"].values.flatten()
print("RD Results")
print(f"  LATE estimate: {rd_samples.mean():.1f} "
      f"[{az.hdi(rd_samples, 0.94)[0]:.1f}, {az.hdi(rd_samples, 0.94)[1]:.1f}]")
print(f"  True LATE: {true_rd_effect}")

## 4. SyntheticControl

**Use when:** One treated unit, multiple untreated donor units, panel data.

**Causal assumption:** The weighted donor combination closely tracks the treated unit before intervention.

In [ ]:
# --- SC: Tobacco tax in one state ---
N_YRS = 20
N_PR = 12  # Pre-intervention years
N_DON = 8
true_sc_effect = -15.0

yrs = np.arange(N_YRS)
Y_don = np.zeros((N_YRS, N_DON))
for j in range(N_DON):
    Y_don[:, j] = np.random.uniform(80, 120) - 1.0 * yrs + np.random.normal(0, 2, N_YRS)

tw = np.zeros(N_DON)
tw[0] = 0.5; tw[2] = 0.3; tw[5] = 0.2
y_tr = Y_don @ tw + np.random.normal(0, 1, N_YRS)
for t in range(N_PR, N_YRS):
    y_tr[t] += true_sc_effect

don_names = [f"State_{j:02d}" for j in range(N_DON)]
sc_rows = []
for t in range(N_YRS):
    sc_rows.append({"unit": "TreatedState", "year": t, "outcome": y_tr[t]})
for j, nm in enumerate(don_names):
    for t in range(N_YRS):
        sc_rows.append({"unit": nm, "year": t, "outcome": Y_don[t, j]})

df_sc = pd.DataFrame(sc_rows)
sc_formula = "outcome ~ 0 + " + " + ".join(don_names)

sc = cp.SyntheticControl(
    data=df_sc,
    treatment_time=N_PR,
    formula=sc_formula,
    group_variable_name="unit",
    treated_group="TreatedState",
    model=cp.pymc_models.WeightedSumFitter(
        sample_kwargs={"draws": 500, "tune": 500, "chains": 2,
                       "progressbar": False, "random_seed": 42}
    ),
)

w_post = sc.idata.posterior["beta"].values.reshape(-1, N_DON)
cf = (w_post @ Y_don.T).mean(axis=0)
gap_post_sc = y_tr[N_PR:] - cf[N_PR:]
gap_hdi_sc = az.hdi((w_post @ Y_don.T)[:, N_PR:] * -1 + y_tr[N_PR:], hdi_prob=0.94)

print("SC Results")
print(f"  Mean post-period gap: {gap_post_sc.mean():.1f}")
print(f"  True effect: {true_sc_effect}")

## Summary: CausalPy API Cheat Sheet

### Common constructor arguments (all models)

```python
model = cp.SomeModel(
    data=df,                        # pandas DataFrame
    formula="y ~ ...",              # formulaic formula string
    model=cp.pymc_models.SomeFitter(
        sample_kwargs={             # passed to pm.sample()
            "draws": 1000,
            "tune": 1000,
            "chains": 4,
            "target_accept": 0.90,
            "random_seed": 42,
        }
    ),
)
```

### Model-specific extra arguments

| Model | Extra arguments |
|-------|----------------|
| `InterruptedTimeSeries` | `treatment_time` |
| `DifferenceInDifferences` | `time_variable_name`, `group_variable_name` |
| `SyntheticControl` | `treatment_time`, `group_variable_name`, `treated_group` |
| `RegressionDiscontinuity` | `running_variable_name`, `treatment_threshold` |

### Accessing results

```python
model.idata                          # ArviZ InferenceData object
model.idata.posterior                # Posterior samples xarray Dataset
model.idata.posterior["beta"]        # Specific parameter samples
az.summary(model.idata)              # Convergence diagnostics table
model.plot()                         # Built-in visualization
```

### Convergence checklist

```python
summary = az.summary(model.idata)
assert summary["r_hat"].max() < 1.01      # R-hat < 1.01
assert summary["ess_bulk"].min() > 400    # ESS > 400
assert model.idata.sample_stats["diverging"].sum() == 0  # No divergences
```